In [ ]:
import pandas as pd
import numpy as np
import re
import string

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss

# Load data 
abstracts = pd.read_csv('abstracts.txt', delimiter='\t', header=None,
                        names=['abstract'])
authors   = pd.read_csv('authors.txt',   delimiter='\t', header=None,
                        names=['authors'])
edgelist  = pd.read_csv('edgelist.txt',  delimiter=',', header=None,
                        names=['source','target'])
test_edges = pd.read_csv('test.txt',     delimiter=',', header=None,
                         names=['source','target'])

# Text preprocessing 
stop_words = set(stopwords.words('english'))
stemmer    = PorterStemmer()

def clean_text(text):
    text = re.sub(r'[^a-zA-Z]', ' ', text).lower()
    tokens = [w for w in text.split() if w not in stop_words]
    stems  = [stemmer.stem(w) for w in tokens if w not in string.punctuation]
    return ' '.join(stems)

abstracts['cleaned'] = abstracts['abstract'].apply(clean_text)

# TF–IDF vectors & cosine-similarity feature
tfidf_vectorizer = TfidfVectorizer(max_features=4000, sublinear_tf=True)
tfidf_matrix     = tfidf_vectorizer.fit_transform(abstracts['cleaned'])
tfidf_vecs       = tfidf_matrix.toarray()

def cosine_similarity(vec1, vec2, eps=1e-10):
    return np.dot(vec1, vec2) / (max(np.linalg.norm(vec1)*np.linalg.norm(vec2), eps))

In [ ]:
# LDA topic modeling prep
count_vec = CountVectorizer(max_features=2000)
dtm       = count_vec.fit_transform(abstracts['cleaned'])
lda       = LatentDirichletAllocation(n_components=25, random_state=42)
topic_dists = lda.fit_transform(dtm)
# topic_dists[i] is the 20-dim topic vector for abstract i

# Negative sampling 
src = edgelist['source'].values
tgt = edgelist['target'].values
forbidden = set(zip(src, tgt)) | set(zip(tgt, src))

n_nodes = abstracts.shape[0]
num_pos = len(edgelist)

negatives = set()
while len(negatives) < num_pos:
    u = np.random.randint(n_nodes)
    v = np.random.randint(n_nodes)
    if u==v or (u,v) in forbidden:
        continue
    negatives.add((u,v))

neg_src, neg_tgt = zip(*negatives)
negative_pairs = pd.DataFrame({
    'source': neg_src,
    'target': neg_tgt,
    'label': 0
})

positive_pairs = edgelist.copy()
positive_pairs['label'] = 1

training_data = pd.concat([positive_pairs, negative_pairs]).sample(frac=1, random_state=42).reset_index(drop=True)

# Jaccard & LDA cosine 
def jaccard(a, b):
    A = set(a.split())
    B = set(b.split())
    return len(A & B) / len(A | B) if len(A|B)>0 else 0.0

# Compute all three features in one pass:
def compute_features(df):
    sims, jacs, lda_sims = [], [], []
    for _, row in df.iterrows():
        i, j = row['source'], row['target']
        # TF-IDF cosine
        sims.append(cosine_similarity(tfidf_vecs[i], tfidf_vecs[j]))
        # Jaccard
        jacs.append(jaccard(abstracts.at[i,'cleaned'], abstracts.at[j,'cleaned']))
        # LDA-topic cosine
        lda_sims.append(cosine_similarity(topic_dists[i], topic_dists[j]))
    df['sim_tfidf']   = sims
    df['sim_jaccard'] = jacs
    df['sim_lda']     = lda_sims

compute_features(training_data)

# Train/Test split & model training
X = training_data[['sim_tfidf','sim_jaccard','sim_lda']]
y = training_data['label']

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train, y_train)

val_preds = model.predict_proba(X_val)[:,1]
print('Validation Log Loss:', log_loss(y_val, val_preds))

In [3]:
# Prepare test set & make submission
compute_features(test_edges)

test_X = test_edges[['sim_tfidf','sim_jaccard','sim_lda']]
test_preds = model.predict_proba(test_X)[:,1]

submission = pd.DataFrame({
    'ID':    np.arange(len(test_edges)),
    'Label': test_preds
})
submission.to_csv('submission.csv', index=False)
print("Submission saved to submission.csv")

Submission saved to submission.csv
